# Cooking Activity — Gaze Signal Metrics

Analyzes gaze signal metrics across all **S7-Cooking** sessions (the largest activity in the dataset with ~154 sessions).

For each session the six signal-level metrics are computed from the preprocessed gaze data:

| Metric | What it captures |
|---|---|
| `mean_yaw_deg` | Average horizontal gaze angle |
| `var_yaw_deg` | Spread of horizontal gaze |
| `mean_pitch_deg` | Average vertical gaze angle |
| `var_pitch_deg` | Spread of vertical gaze |
| `mean_depth_m` | Average viewing distance |
| `var_depth_m` | Spread of viewing distance |

In [8]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [9]:
from pathlib import Path
import pandas as pd
import nymeria_gaze_tools as ngt

DATA_ROOT = Path("../data/processed")

In [10]:
catalog          = ngt.load_metadata(data_root=DATA_ROOT)
cooking_sessions = ngt.filter_sessions(catalog, script="S7-Cooking", has_gaze_data=True)

print(f"{len(cooking_sessions)} cooking sessions found")

154 cooking sessions found


In [11]:
rows = []

for i, (_, row) in enumerate(cooking_sessions.iterrows(), start=1):
    uid = row["sequence_uid"]
    print(f"[{i}/{len(cooking_sessions)}] {uid}")
    try:
        raw     = ngt.load_session(uid, data_root=DATA_ROOT)
        df      = ngt.preprocess(raw)
        metrics = ngt.gaze_signal_metrics(df)
        metrics["sequence_uid"] = uid
        rows.append(metrics)
    except Exception as e:
        print(f"  Skipping {uid}: {e}")

summaries = pd.DataFrame(rows)
print(f"\n{len(summaries)} sessions loaded")
print(summaries[["mean_yaw_deg", "var_yaw_deg", "mean_pitch_deg", "var_pitch_deg", "mean_depth_m", "var_depth_m"]].describe().round(3))

[1/154] 20230628_s0_adriana_gonzalez_act4_rzxlmo
[2/154] 20230929_s1_alan_burns_act4_zch0c7
[3/154] 20231010_s0_albert_hammond_act3_wshdw5
[4/154] 20230825_s1_alejandra_reynolds_act1_5i0rcc
[5/154] 20231121_s0_alexandria_griffith_act0_945arb
[6/154] 20230801_s1_alexis_hernandez_act0_abfhom
[7/154] 20231205_s1_alexis_wyatt_act0_0e9t84
[8/154] 20231205_s1_alexis_wyatt_act1_sqkh5t
[9/154] 20231025_s1_alicia_drake_act3_dja4ib
[10/154] 20230823_s1_alison_riddle_act0_xf9eze
[11/154] 20231003_s1_allen_evans_act4_cv3qh0
[12/154] 20231030_s1_allison_harris_act3_fe2ruo
[13/154] 20231106_s1_amanda_rodgers_act4_kli4b8
[14/154] 20231220_s1_amanda_wong_act0_0kpd1x
[15/154] 20231220_s1_amanda_wong_act1_pumk92
[16/154] 20230713_s1_amy_crawford_act0_gnlv1f
[17/154] 20230818_s0_amy_padilla_act4_dq1iuq
[18/154] 20231026_s1_amy_rosales_act3_8zoku8
[19/154] 20231026_s1_amy_rosales_act4_m0nqli
[20/154] 20231107_s0_amy_snow_act3_y044ox
[21/154] 20231115_s1_andrew_johnson_act4_rhpi50
[22/154] 20230829_s1_ange

In [12]:
age_lookup = cooking_sessions.set_index("sequence_uid")["participant_age_group"]
summaries["age_group"] = summaries["sequence_uid"].map(age_lookup)
print(summaries["age_group"].value_counts().sort_index())

age_group
18-24    34
25-30    48
31-35    32
36-40    20
41-45    11
46-50     9
Name: count, dtype: int64


---
## 1 · Overview — all sessions

In [13]:
all_sessions = {"All sessions": [summaries]}
for col in ["mean_yaw_deg", "mean_pitch_deg", "mean_depth_m",
            "var_yaw_deg",  "var_pitch_deg",  "var_depth_m"]:
    ngt.plot_gaze_position_boxplots(all_sessions, column=col).show()

---
## 2 · By Age Group

In [14]:
age_order = sorted(summaries["age_group"].dropna().unique())
groups = {age: [summaries[summaries["age_group"] == age]] for age in age_order}

for col, label in [
    ("mean_yaw_deg",   "Mean Yaw"),
    ("mean_pitch_deg", "Mean Pitch"),
    ("mean_depth_m",   "Mean Depth"),
    ("var_yaw_deg",    "Yaw Variance"),
    ("var_pitch_deg",  "Pitch Variance"),
    ("var_depth_m",    "Depth Variance"),
]:
    ngt.plot_gaze_position_boxplots(
        groups, column=col,
        title=f"{label} by Age Group — S7-Cooking",
    ).show()